# Phase 1: one week of inference experiments
Three dedicated VMs (`vllm`, `sglang`, `trtllm`) and one sequential cross-framework VM; one GPU each, 672 GPU-hours total. Use identical pinned model revisions and workloads when comparing frameworks.

Install the repository with `pip install -e '.[notebook]'` in each framework-compatible environment. The notebook runs selected cells interactively; the matching Python script supervises the complete queue. CPU simulation is the default. Set `RUN_REAL_GPU=True` only after filling verified preflight records. Simulation results are not GPU measurements.


## Differences from the v2 lab
See [the parity checklist](phase1_v2_parity.md). Independent optional cells below restore Pareto/constraint searches, additional pruners, calibration diagnostics, and pinned HF/speculative setup. The supervised default scalar queue remains unchanged. Untuned baselines and full v2 reporting remain separate. Phase A defaults to the same target as B; explicit HF setup changes only Phase B.


In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "llm_optimizer").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from llm_optimizer.experiment.phase1 import Phase1Runner, plot_group


/home/ubuntu/.venv-vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## VM, model, workload, and budget controls
Change `FRAMEWORKS` to a single framework on dedicated VMs, or all three on the comparison VM. All frameworks on one VM run sequentially and must be installed compatibly. Use a separate campaign name for independent replications. Changing any search setting creates a new configuration hash; restarts with identical settings resume existing studies. All campaigns on a VM must share `RUN_ROOT` and the same GPU assignment.
`SAMPLER_NAMES` selects GP (`gp`), TPE (`tpe`), and multivariate TPE (`tpe_multivariate`). GP requires the engine-compatible PyTorch installation; unavailable GP is explicitly reported and recorded as skipped, never replaced by TPE. Each combination of sampler, seed, framework, concurrency, objective and pruner has **20 attempted trials** by default. Failed/pruned attempts count toward that budget. Three samplers and three seeds therefore mean 180 attempts for a single concurrency/objective/framework/pruner, not 20 total. Optional `sampler_names=["gp"]` on a group call restricts that cell.

| Concurrency | Requests per trial | Default trial attempts per study |
|---|---|---|
| 1 | 32 | 20 |
| 8 | 62 | 20 |
| 16 | 125 | 20 |
| 32 | 250 | 20 |
| 128 | 1000 | 20 |

The loaded request/concurrency ratio is approximately 1000/128. Concurrency 1 deliberately uses 32 requests. `REQUESTS_BY_CONCURRENCY` is used by both A and B; explicit validation counts remain independently configured at 1000. Pruning observation floors scale from v2's 400/1000 ratio: 12, 24, 50, 100, 400 respectively. With small request counts, tail-latency estimates are noisy; the separate validation budget remains necessary.


In [22]:
CAMPAIGN = "phase1-week1"
FRAMEWORKS = ["vllm"]  # or ["vllm", "sglang", "trtllm"]
RUN_REAL_GPU = True
CONCURRENCIES = [1, 8, 16, 32, 128]
SEEDS = [11, 29, 47]
RUN_ROOT = ROOT / "experiment_results" / "phase1"
CAMPAIGN_HOURS = 168
GROUP_TIMEOUT_SECONDS = 6 * 3600  # includes all frameworks/seeds in a group
MAX_RESTARTS = 2  # bounded retries after a crashed/timed-out group
MODEL = "Qwen/Qwen3.5-4B"  # same model across A/B isolates search-space effects
MODEL_SNAPSHOT = "/workspace/models/pinned-model-snapshot"
TRIALS = 20  # attempts PER sampler/seed/framework/concurrency/pruner
SAMPLER_NAMES = ["tpe", "tpe_multivariate"]  # select any nonempty subset
PARETO_SAMPLERS = [*SAMPLER_NAMES, "nsga2"]  # available to optional Pareto calls
REQUESTS_BY_CONCURRENCY = {1: 32, 8: 62, 16: 125, 32: 250, 128: 1000}
# Preserve the v2 400/1000 pruning-observation fraction at each request budget.
PRUNING_MIN_COMPLETED_REQUESTS = {
    c: max(1, int(n * 0.4)) for c, n in REQUESTS_BY_CONCURRENCY.items()
}
SAMPLER_SETTINGS = {
    "startup_single": 4, "startup_multi": 10,
    "n_ei_candidates": 24, "population_size": 8,
}
VALIDATION_REPEATS = 5

# W&B controls match optuna_inference_optimization_lab_v2.
WANDB_ENABLED = True
WANDB_PROJECT = "inference-optimization"
WANDB_ENTITY = "maorblumberg-tel-aviv-university"  # or None for your SDK default
WANDB_MODE = "online"  # "offline" keeps runs local; False above disables the SDK
WANDB_GROUP = CAMPAIGN  # same label across framework VMs

# Fill with actual verified values. Never mark checks true without performing them.
PREFLIGHT = {
    framework: {
        "gpu_type": "REPLACE", "framework_version": "REPLACE",
        "cuda_version": "REPLACE", "driver_version": "REPLACE",
        "model_revision": "REPLACE_WITH_MODEL_COMMIT",
        "native_config_verified": True, "quality_verified": False,
        "verified_precision_combinations": [],  # weights/cache pairs approved on this VM
    }
    for framework in ["vllm", "sglang", "trtllm"]
}


## W&B login and study logging
Install the SDK with `pip install -e '.[notebook,wandb]'`. Run the login cell before creating studies, as in `optuna_inference_optimization_lab_v2.ipynb`. Use `WANDB_ENABLED=False` to disable tracking, or `WANDB_MODE="offline"` to save runs locally without login; upload offline runs later with `wandb sync`.

The existing shared study logger records sampled parameters (including precision), rendered engine configs, workload/environment contracts, benchmark/objective metrics, request accounting, failure/pruning details, cumulative GPU cost, and final best/Pareto candidates. One W&B run is created per study invocation, named after its stable Optuna study name and grouped by `WANDB_GROUP`. Synthetic runs carry the `SYNTHETIC` tag. W&B logging failures warn and allow optimization to continue. Independent validation remains in local artifacts, matching v2.

For unattended runs, authenticate on the VM first with `wandb login` or provide `WANDB_API_KEY` through the service environment. Keep credentials outside notebook cells and experiment configs. The batch supervisor authenticates before launching the queue; CPU smoke tests disable tracking.


In [4]:
if WANDB_ENABLED and WANDB_MODE == "online":
    import wandb
    wandb.login()  # uses existing credentials/WANDB_API_KEY or prompts interactively


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/ubuntu/.netrc.
wandb: Currently logged in as: maorblumberg (maorblumberg-tel-aviv-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


## Explicit base and extended search spaces
Every domain below is editable. Base numeric domains must be regularly spaced and contain at least two values. Extended categorical domains must be supported by the installed engine and model. Remove unsupported choices after preflight, including numerical quality checks for cache dtype changes. Graph profiles and padding are conditional. `schedule_conservativeness` is SGLang-only; Phase A retains its fixed baseline value of 1.0, while Phase B searches the explicit domain below.

In [7]:
# Illustrative SLOs; replace after measuring the pinned model/workload/GPU.
TTFT_LIMITS_MS = {"1": 100.0, "128": 200.0}
E2E_LIMITS_MS = {"1": 2000.0, "128": 6000.0}
FAILURE_RATE_LIMIT = 0.0
CONSTRAINED = False
PRUNER_NAMES = ["nop", "median", "percentile", "halving", "hyperband", "threshold"]
PRUNING_SAMPLES = 32
# Ordinary serving is explicit. Setup is opt-in and may download checkpoints.
HF_SETUP = {
    "target_model_id": MODEL, "target_revision": "REPLACE_WITH_MODEL_COMMIT",
    "framework": "sglang", "enable_dflash": False,
    "speculative_algorithm": None, "draft_model_id": None, "draft_revision": None,
    "block_size": 16, "num_speculative_tokens": 15, "max_draft_len": 3,
}
# SGLang: DFLASH/DSPARK; vLLM: enable_dflash=True selects DSpark.
# TRT: explicit algorithms accepted by predefined.trtllm.validate_draft_request.
# Configure a separate campaign/run root to compare ordinary and speculative serving.
MODEL_PROFILES = {}  # populated only by explicit setup_pinned_models()
MODEL_SETUP_IDENTITY = None
MODEL_SETUP_CHOICES = None
PHASE_B_MODEL = None
PHASE_B_SNAPSHOT = None


In [8]:
PREFILL_BUDGETS = list(range(1024, 8193, 1024))
TOKEN_BUDGETS = list(range(1024, 32769, 256))
CAPACITIES = list(range(8, 129, 8))
BASE_SPACES = {
    1: {
        "vllm": {"max_num_batched_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "sglang": {"chunked_prefill_size": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
        "trtllm": {"max_num_tokens": PREFILL_BUDGETS, "cuda_graph_on": [0, 1]},
    },
    128: {  # shared regime for all concurrency levels > 1
        "vllm": {
            "max_num_seqs": CAPACITIES, "max_num_batched_tokens": TOKEN_BUDGETS,
            "gpu_memory_utilization": [0.75, 0.80, 0.85, 0.90],
        },
        "sglang": {
            "max_running_requests": CAPACITIES, "chunked_prefill_size": TOKEN_BUDGETS,
            "mem_fraction_static": [0.70, 0.75, 0.80, 0.85],
        },
        "trtllm": {
            "max_batch_size": CAPACITIES, "max_num_tokens": TOKEN_BUDGETS,
            "kv_cache_free_gpu_memory_fraction": [0.65, 0.75, 0.85, 0.95],
        },
    },
}

EXPANDED_CHOICES = {'vllm': {'async_scheduling': [False, True],
          'performance_mode': ['balanced', 'interactivity', 'throughput'],
          'attention_backend': ['auto', 'FLASH_ATTN', 'FLASHINFER'],
          'kv_cache_dtype': ['auto'],
          'enable_prefix_caching': [False, True],
          'graph_mode': ['NONE', 'PIECEWISE', 'FULL_AND_PIECEWISE']},
 'sglang': {'attention_backend': ['triton', 'fa3', 'flashinfer'],
            'disable_radix_cache': [False, True],
            'schedule_policy': ['fcfs', 'lpm'],
            'enable_mixed_chunk': [False, True],
            'disable_overlap_schedule': [False, True],
            'cuda_graph_backend_decode': ['disabled', 'full'],
            'cuda_graph_backend_prefill': ['disabled', 'breakable']},
 'trtllm': {'attn_backend': ['TRTLLM'],
            'enable_chunked_prefill': [False, True],
            'enable_block_reuse': [False, True],
            'capacity_scheduler_policy': ['GUARANTEED_NO_EVICT', 'MAX_UTILIZATION'],
            'graph_enabled': [False, True]}}

EXTRA_SPACES = {
    "vllm": {"graph_capture_profile": ["powers", "dense"]},
    "sglang": {
        "schedule_conservativeness": [0.5, 0.75, 1.0, 1.25, 1.5],
        "max_prefill_tokens": [8192, 16384, 24576, 32768],
    },
    "trtllm": {
        "graph_capture_profile": ["powers", "dense"],
        "graph_padding": [0, 1],
    },
}


## Bind the shared runner
Persistent SQLite storage is local to each VM. Copy artifacts off the VM periodically. A VM-wide file lock prevents overlapping workers under the same root. The runner records abandoned RUNNING trials as FAIL after acquiring exclusive ownership; failed and pruned attempts count toward the configured trial budget. Notebook kernels do not have the batch script’s process-group watchdog. After an interrupted notebook, ensure its server processes have stopped before resuming.
Every group invocation reads the current controls and resumes via SQLite. Editing the request map or sampler selection produces a separate campaign configuration hash.


## Weight precision and KV-cache precision
Phase B searches the two axes independently: BF16/BF16, BF16/FP8, FP8/BF16, and FP8/FP8. Set either domain to `["fp8"]` to force FP8. `dtype` remains `bfloat16` for compute/non-quantized layers: FP8 weights use quantization, not `dtype="fp8"`. This uses online/on-load quantization from the same **unquantized pinned source checkpoint**, so models and workloads stay comparable; it does not generate or persist an FP8 weight checkpoint. “Cache” here means the runtime KV cache, not a disk/download cache.

Mappings: vLLM/SGLang use `quantization="fp8"` for FP8 weights (omit for BF16). vLLM uses `kv_cache_dtype="fp8"` plus calculated KV scales; SGLang uses `fp8_e4m3`; BF16 cache uses `auto` with BF16 dtype. TensorRT-LLM uses YAML `quant_config.quant_algo` and `quant_config.kv_cache_quant_algo`, with `FP8` or null, plus calibration settings for FP8 weights. Its KV-cache configuration also explicitly selects `fp8` or the default dtype. Remove incompatible attention backends or other options before launch.

On each GPU VM, verify backend support, calibration/scaling, and output quality for every selected combination and the installed version/model/GPU. Then list the approved pairs in `PREFLIGHT[framework]["verified_precision_combinations"]`, e.g. `["bf16/bf16", "bf16/fp8", "fp8/bf16", "fp8/fp8"]`. The runner checks this list; it does not run quality evaluation itself. Calibration startup counts toward allocation time. CPU simulation only verifies wiring and cannot estimate the performance/accuracy benefit of FP8 weights.

Framework references: [vLLM FP8](https://docs.vllm.ai/en/latest/features/quantization/llm_compressor/fp8/), [SGLang quantization](https://docs.sglang.io/docs/advanced_features/quantization), [SGLang KV cache](https://github.com/sgl-project/sglang/blob/main/docs/docs/advanced_features/quantized_kv_cache.mdx), [TensorRT-LLM FP8](https://nvidia.github.io/TensorRT-LLM/1.1.0rc1/performance/performance-tuning-guide/fp8-quantization.html). Verify schemas against the pinned serving versions.


In [9]:
# Independent Phase B axes. To force FP8 weights + FP8 cache, use ["fp8"] for both.
# Phase A always uses unquantized BF16 weights and the default cache (BF16).
PRECISION_CHOICES = {
    "vllm": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "sglang": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
    "trtllm": {"weight_precision": ["bf16", "fp8"], "kv_cache_precision": ["bf16", "fp8"]},
}
# TRT on-load weight quantization needs calibration. Adjust for your pinned version;
# prepare/cache its calibration dataset before launching the unattended campaign.
TRT_FP8_CALIBRATION = {
    "calib_batches": 32, "calib_batch_size": 1,
    "calib_max_seq_length": 512, "tokenizer_max_seq_length": 512,
}


In [ ]:
# Notebook-only vLLM 0.33 adapter; run this, the runner definition, and the adapter-binding cell in order.
from types import MethodType
from llm_optimizer.experiment.phase1 import ExplicitLab

VLLM033_COMPATIBILITY = {
    "adapter": "notebook-vllm033-v2",
    "online_weight_scheme": "fp8_per_tensor",
    "fp8_kv_scaling": "unit-scales-unless-checkpoint-calibrated",
}

def notebook_precision_config(self, framework, weights, cache):
    config = ExplicitLab.precision_config(self, framework, weights, cache)
    if framework == "vllm":
        config.pop("calculate_kv_scales", None)
        if weights == "fp8":
            config["quantization"] = VLLM033_COMPATIBILITY["online_weight_scheme"]
    return config

# The repository builds typed Args before launching the command. Register the
# new spelling in memory so both real benchmarks and the command audit accept it.
from llm_optimizer.args import ArgConfig
from llm_optimizer.predefined import SERVER_CONFIGS
SERVER_CONFIGS["vllm"]["disable_prefix_caching"] = ArgConfig[bool](name="disable_prefix_caching")

def notebook_serialize_engine_config(self, framework, concurrency, config):
    native = ExplicitLab.serialize_engine_config(self, framework, concurrency, config)
    if framework == "vllm" and "no_enable_prefix_caching" in native:
        native["disable_prefix_caching"] = native.pop("no_enable_prefix_caching")
    return native


In [12]:
def make_runner(for_model_setup=False):
    if MODEL_SETUP_IDENTITY is not None and MODEL_SETUP_IDENTITY != HF_SETUP:
        raise ValueError("HF_SETUP changed; clear MODEL_PROFILES and MODEL_SETUP_IDENTITY, then rerun setup")
    if MODEL_SETUP_CHOICES is not None and MODEL_SETUP_CHOICES != EXPANDED_CHOICES[HF_SETUP["framework"]]:
        raise ValueError("Expanded choices changed after HF setup; clear setup profiles and rerun setup")
    settings = {
        "ttft_limits_ms": TTFT_LIMITS_MS, "e2e_limits_ms": E2E_LIMITS_MS,
        "failure_rate_limit": FAILURE_RATE_LIMIT, "pruning_samples": PRUNING_SAMPLES,
        "constrained": CONSTRAINED,
        "wandb_enabled": WANDB_ENABLED,
        "wandb_project": WANDB_PROJECT, "wandb_entity": WANDB_ENTITY,
        "wandb_mode": WANDB_MODE, "wandb_group": WANDB_GROUP,
        "frameworks": FRAMEWORKS, "real_frameworks": FRAMEWORKS,
        "run_real_gpu": RUN_REAL_GPU and not for_model_setup, "quick_demo": False,
        "phases": ["A", "B"], "concurrencies": CONCURRENCIES,
        "seeds": SEEDS, "sampler_names": SAMPLER_NAMES, "pareto_samplers": PARETO_SAMPLERS,
        "phase_models": {"A": MODEL, "B": PHASE_B_MODEL or MODEL},
        "model_snapshots": {"A": MODEL_SNAPSHOT, "B": PHASE_B_SNAPSHOT or MODEL_SNAPSHOT},
        # Lab's phase-level fallback; actual study counts use the map below.
        "phase_requests": {"A": REQUESTS_BY_CONCURRENCY[128],
                           "B": REQUESTS_BY_CONCURRENCY[128]},
        "sampler_settings": SAMPLER_SETTINGS,
        "search_trials": {p: {"1": TRIALS, "128": TRIALS} for p in ["A", "B"]},
        "base_workload": {
            "dataset_name": "random", "apply_chat_template": True, "random_input_len": 512,
            "random_output_len": 128, "random_range_ratio": 1.0,
            "seed": 2026, "request_rate": "inf",
        },
        "validation_repeats": VALIDATION_REPEATS,
        "validation_requests": {"A": 1000, "B": 1000},
        "gpu_count": 1, "rest_seconds": 10,
        "real_preflight": {
            phase: {f: {**PREFLIGHT[f], "expanded_choices": EXPANDED_CHOICES[f],
                           **(MODEL_PROFILES.get(f, {}) if phase == "B" else {})}
                    for f in FRAMEWORKS}
            for phase in ["A", "B"]
        },
    }
    return Phase1Runner(
        settings, BASE_SPACES, EXPANDED_CHOICES, EXTRA_SPACES, RUN_ROOT,
        precision_choices=PRECISION_CHOICES, trt_fp8_calibration=TRT_FP8_CALIBRATION,
        requests_by_concurrency=REQUESTS_BY_CONCURRENCY,
        pruning_min_completed_requests=PRUNING_MIN_COMPLETED_REQUESTS,
    )

# Construct from the current controls for each invocation; SQLite resumes studies.
def get_runner():
    return make_runner()

def run_objective_group(phase, target, concurrencies, pruner_name="nop", sampler_names=None, constrained=None):
    return get_runner().run_objective_group(
        phase, target, concurrencies, pruner_name, sampler_names=sampler_names,
        constrained=CONSTRAINED if constrained is None else constrained,
    )

def setup_pinned_models():
    """Explicit GPU setup; commit resolved profiles into the editable controls."""
    global PHASE_B_MODEL, PHASE_B_SNAPSHOT, MODEL_SETUP_IDENTITY, MODEL_SETUP_CHOICES
    if not RUN_REAL_GPU:
        raise ValueError("Set RUN_REAL_GPU=True before model setup")
    runner = make_runner(for_model_setup=True)
    runner.lab.RUN_REAL_GPU = True
    with runner.lock():
        summary = runner.lab.configure_hf_models(**HF_SETUP)
    framework = HF_SETUP["framework"]
    import copy
    profile = runner.lab.REAL_PREFLIGHT["B", framework]
    # Verification declarations remain editable in PREFLIGHT after setup.
    MODEL_PROFILES[framework] = {
        key: copy.deepcopy(profile[key]) for key in (
            "model_revision", "hf_target_model_id", "fixed_server", "draft_model_id",
            "draft_revision", "speculative_decoding", "framework_version", "cuda_version",
            "driver_version", "gpu_type",
        ) if key in profile
    }
    if profile["expanded_choices"] != EXPANDED_CHOICES[framework]:
        MODEL_PROFILES[framework]["expanded_choices"] = copy.deepcopy(profile["expanded_choices"])
    MODEL_SETUP_CHOICES = copy.deepcopy(EXPANDED_CHOICES[framework])
    PHASE_B_MODEL, PHASE_B_SNAPSHOT = summary["target_model_id"], summary["target_path"]
    MODEL_SETUP_IDENTITY = dict(HF_SETUP)
    return summary


In [ ]:
# Attach the adapter to every notebook-created runner, including model setup.
# Preserve the original factory so this cell is safe to rerun.
_original_make_runner = getattr(make_runner, "_without_vllm033_adapter", make_runner)

def make_runner(for_model_setup=False):
    if "vllm" in FRAMEWORKS:
        # Persist the behavior change in the campaign hash before construction.
        PREFLIGHT["vllm"]["notebook_compatibility"] = dict(VLLM033_COMPATIBILITY)
    runner = _original_make_runner(for_model_setup=for_model_setup)
    runner.lab.precision_config = MethodType(notebook_precision_config, runner.lab)
    runner.lab.serialize_engine_config = MethodType(notebook_serialize_engine_config, runner.lab)
    return runner

make_runner._without_vllm033_adapter = _original_make_runner


## Resolve a pinned Hugging Face model for vLLM
Run these cells after the configuration and runner cells. Setup downloads or reuses the model snapshot and requires a CUDA GPU. Choose an immutable commit for reproducible reruns; `main` resolves to its current commit. The setup below selects ordinary serving.


In [15]:
import tempfile
from pathlib import Path

hf_cache = Path(tempfile.mkdtemp(prefix="hf-cache-", dir="/tmp"))
HF_SETUP["cache_dir"] = str(hf_cache)

print("Hugging Face cache:", hf_cache)
print("Writable:", hf_cache.is_dir())

Hugging Face cache: /tmp/hf-cache-08jual1n
Writable: True


In [28]:
FRAMEWORKS = ["vllm"]
RUN_REAL_GPU = True
MODEL_PROFILES = {}
MODEL_SETUP_IDENTITY = None
MODEL_SETUP_CHOICES = None

HF_SETUP.update(
    framework="vllm", target_model_id=MODEL, target_revision="main",
    enable_dflash=False, speculative_algorithm=None,
    draft_model_id=None, draft_revision=None,
)
hf_models = setup_pinned_models()
hf_models


Fetching 14 files: 100%|███████████████████████| 14/14 [00:00<00:00, 2500.44it/s]

Model setup complete for vllm. Run the existing Phase B objective cells.


{'framework': 'vllm',
 'target_model_id': 'Qwen/Qwen3.5-4B',
 'target_revision': '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a',
 'target_path': '/tmp/hf-cache-08jual1n/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a',
 'draft_model_id': None,
 'draft_revision': None,
 'draft_path': None,
 'speculative_decoding': None}

In [29]:
MODEL = hf_models["target_model_id"]
MODEL_SNAPSHOT = hf_models["target_path"]
for key in ("gpu_type", "framework_version", "cuda_version",
            "driver_version", "model_revision"):
    PREFLIGHT["vllm"][key] = MODEL_PROFILES["vllm"][key]
print("Pinned snapshot:", MODEL_SNAPSHOT)
print("Pinned revision:", PREFLIGHT["vllm"]["model_revision"])


Pinned snapshot: /tmp/hf-cache-08jual1n/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
Pinned revision: 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


In [30]:
MODEL = hf_models["target_model_id"]
MODEL_SNAPSHOT = hf_models["target_path"]

# Record the resolved runtime and revision for Phase A.
for key in (
    "gpu_type",
    "framework_version",
    "cuda_version",
    "driver_version",
    "model_revision",
):
    PREFLIGHT["vllm"][key] = MODEL_PROFILES["vllm"][key]

print("Phase A snapshot:", MODEL_SNAPSHOT)
print("Phase B snapshot:", PHASE_B_SNAPSHOT)

Phase A snapshot: /tmp/hf-cache-08jual1n/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
Phase B snapshot: /tmp/hf-cache-08jual1n/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


## Notebook-only vLLM 0.33 compatibility check
Run after pinned-model setup and before the verification declarations. The adapter removes
`calculate_kv_scales`, maps online FP8 weights to `fp8_per_tensor`, and translates
`no_enable_prefix_caching` to `disable_prefix_caching` in the native serializer.
The latter spelling is registered in the kernel's argument registry for real benchmarks too. It is attached to each
notebook runner instance; batch Python entry points do not inherit it. Compatibility metadata
is included in the frozen campaign configuration, creating a different study directory from
the failed campaign. Keep the same metadata when resuming this campaign.

[vLLM 0.33 online quantization](https://docs.vllm.ai/en/v0.33.0/features/quantization/online/)
documents this weight scheme. [FP8 KV cache](https://docs.vllm.ai/en/v0.33.0/features/quantization/quantized_kvcache/)
uses scales of 1.0 without checkpoint calibration. Removing the old flag does **not** provide
dynamic scale calibration. Inspect generated text against a BF16 reference before approving FP8.
The CLI audit checks flag names; it cannot prove GPU/backend/graph combinations work.


In [ ]:
import re
import shutil
import subprocess
from importlib.metadata import version

VLLM_EXECUTABLE = shutil.which("vllm")
if VLLM_EXECUTABLE is None:
    raise RuntimeError("vllm is absent from this kernel's PATH")
installed_vllm = version("vllm")
if not installed_vllm.startswith("0.33."):
    raise RuntimeError(f"This adapter targets vLLM 0.33.x, found {installed_vllm}")
cli_version = subprocess.run([VLLM_EXECUTABLE, "--version"], capture_output=True,
                             text=True, check=True, timeout=120)
if installed_vllm not in cli_version.stdout + cli_version.stderr:
    raise RuntimeError("Kernel package and PATH vllm versions differ")
help_result = subprocess.run([VLLM_EXECUTABLE, "serve", "--help=all"],
                             capture_output=True, text=True, check=True, timeout=120)
VLLM_SERVE_OPTIONS = set(re.findall(r"--[a-z][a-z0-9-]*", help_result.stdout))
print("Kernel:", sys.executable, "CLI:", VLLM_EXECUTABLE, "version:", installed_vllm)
print("Compatibility:", VLLM033_COMPATIBILITY)

def notebook_vllm_command(lab, config, port=8000):
    import shlex
    # Same native serialization used by the benchmark execution path.
    native = lab.serialize_engine_config("vllm", 8, config)
    arguments = [VLLM_EXECUTABLE, "serve", str(MODEL_SNAPSHOT),
                 "--host", "127.0.0.1", "--port", str(port)]
    for key, value in native.items():
        flag = "--" + key.replace("_", "-")
        if flag not in VLLM_SERVE_OPTIONS:
            raise ValueError(f"Installed vLLM does not advertise {flag}")
        arguments.append(flag if value is True else f"{flag}={value}")
    return shlex.join(arguments)

# Audit every precision mapping without creating Optuna trials or starting a server.
audit_runner = make_runner(for_model_setup=True)
for weights in PRECISION_CHOICES["vllm"]["weight_precision"]:
    for cache in PRECISION_CHOICES["vllm"]["kv_cache_precision"]:
        config = audit_runner.lab.precision_config("vllm", weights, cache)
        print(weights, cache, notebook_vllm_command(audit_runner.lab, config))

# Audit base and extended flag names as well (graph settings are nested JSON).
audit_base = audit_runner.lab.render_engine_config("vllm", 8, {
    "max_num_seqs": 8, "max_num_batched_tokens": 2048,
    "gpu_memory_utilization": 0.75,
})
audit_base.update(MODEL_PROFILES["vllm"].get("fixed_server", {}))
print("Base:", notebook_vllm_command(audit_runner.lab, audit_base))
for name, values in EXPANDED_CHOICES["vllm"].items():
    for value in values:
        config = ({"compilation_config": {"cudagraph_mode": value}}
                  if name == "graph_mode" else {name: value})
        notebook_vllm_command(audit_runner.lab, config)
print("All selected base/extended flag names are advertised by the installed CLI.")


In [ ]:
# Optional: starts ONE real server, requests text, then stops it; no Optuna trial.
RUN_VLLM033_SMOKE = False
SMOKE_WEIGHTS, SMOKE_CACHE = "fp8", "fp8"  # repeat with bf16/bf16 as a reference
SMOKE_PORT = 8000  # must be unused
if RUN_VLLM033_SMOKE:
    import requests
    from llm_optimizer.server_utils import start_server, terminate_process_top_down

    smoke_runner = make_runner(for_model_setup=True)
    smoke_lab = smoke_runner.lab
    smoke_config = smoke_lab.render_engine_config("vllm", 8, {
        "max_num_seqs": 8, "max_num_batched_tokens": 2048,
        "gpu_memory_utilization": 0.75,
    })
    smoke_config.update(smoke_lab.precision_config("vllm", SMOKE_WEIGHTS, SMOKE_CACHE))
    # Start with a small eager configuration and automatic attention selection.
    # Test the expanded backends/graphs/scheduling separately before enabling those domains.
    smoke_config.update(attention_backend="auto", enforce_eager=True,
                        async_scheduling=False, enable_prefix_caching=False)
    smoke_config.pop("compilation_config", None)
    smoke_config = {**MODEL_PROFILES["vllm"].get("fixed_server", {}), **smoke_config}
    smoke_command = notebook_vllm_command(smoke_lab, smoke_config, SMOKE_PORT)
    print(smoke_command)
    with smoke_runner.lock():
        process = None
        try:
            process = start_server(smoke_command, {},
                                   f"http://127.0.0.1:{SMOKE_PORT}/health", False)
            response = requests.post(f"http://127.0.0.1:{SMOKE_PORT}/v1/chat/completions",
                json={"model": str(MODEL_SNAPSHOT), "messages": [
                    {"role": "user", "content": "What is 17 times 19? Explain briefly."}],
                    "temperature": 0, "max_tokens": 256}, timeout=180)
            response.raise_for_status()
            print(response.json()["choices"][0]["message"])
        finally:
            if process is not None:
                terminate_process_top_down(process)
    print("Smoke completed. Review text; this does not approve the expanded search space.")


Set the native configuration and quality verification fields in `PREFLIGHT["vllm"]` only after checking them on this VM. List every verified weight/cache combination selected by `PRECISION_CHOICES`; narrow those choices if needed. Run the experiment cells after setup. Rerunning the initial controls resets the setup, so run these cells again. Notebook variables are not passed to the batch script.


In [ ]:
print("vLLM version:", MODEL_PROFILES["vllm"]["framework_version"])
print("Expanded choices:", EXPANDED_CHOICES["vllm"])

# Select the precision combinations you intend to test.
PRECISION_CHOICES["vllm"] = {
    "weight_precision": ["fp8"],   # e.g. ["bf16", "fp8"]
    "kv_cache_precision": ["fp8"], # e.g. ["bf16", "fp8"]
}

# Change these to True only after checking native configuration support
# and testing that the model's generated output is acceptable.
PREFLIGHT["vllm"].update({
    "native_config_verified": False,
    "quality_verified": False,

    # List only combinations you verified on this VM.
    # Names use "weight_precision/kv_cache_precision".
    "verified_precision_combinations": [
        # "bf16/bf16",
        # "bf16/fp8",
        # "fp8/bf16",
        # "fp8/fp8",
    ],
})

print("Preflight:", PREFLIGHT["vllm"])
print("Precision choices:", PRECISION_CHOICES["vllm"])

## Independent experiment cells
Day 1: preflight, untuned baseline measurements, and recovery drills. Days 2–4: base (A) versus extended (B) searches on the same model, then median versus no pruning. Days 5–6: independent repeats and workload checks. Day 7: sustained-load confirmation. The queue below performs the search stage; use the validation cell afterwards. Measure trial duration during the pilot and adjust trial counts to fit the week. A finite completed queue exits successfully rather than repeating identical experiments indefinitely.

### A: single e2e (no pruning)

In [ ]:
A_single_e2e_studies = run_objective_group(
    "A", "e2e", [1], pruner_name="nop",
)
plot_group(A_single_e2e_studies)


### A: single ttft (no pruning)

In [ ]:
A_single_ttft_studies = run_objective_group(
    "A", "ttft", [1], pruner_name="nop",
)
plot_group(A_single_ttft_studies)


### A: loaded throughput (no pruning)

In [ ]:
A_loaded_throughput_studies = run_objective_group(
    "A", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_throughput_studies)


### A: loaded e2e (no pruning)

In [ ]:
A_loaded_e2e_studies = run_objective_group(
    "A", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(A_loaded_e2e_studies)


### B: single e2e (no pruning)

In [ ]:
B_single_e2e_studies = run_objective_group(
    "B", "e2e", [1], pruner_name="nop",
)
plot_group(B_single_e2e_studies)


### B: single ttft (no pruning)

In [ ]:
B_single_ttft_studies = run_objective_group(
    "B", "ttft", [1], pruner_name="nop",
)
plot_group(B_single_ttft_studies)


### B: loaded throughput (no pruning)

In [ ]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_throughput_studies)


### B: loaded e2e (no pruning)

In [ ]:
B_loaded_e2e_studies = run_objective_group(
    "B", "e2e", [c for c in CONCURRENCIES if c > 1], pruner_name="nop",
)
plot_group(B_loaded_e2e_studies)


### B: loaded throughput with median pruning
The first ten trials establish pruning history. Decisions use measured request checkpoints, with at least three reference trials. The no-pruning comparison above uses the same workload and seeds.

In [39]:
B_loaded_throughput_studies = run_objective_group(
    "B", "throughput", [c for c in CONCURRENCIES if c > 1],
    pruner_name="median",
)
plot_group(B_loaded_throughput_studies)


/tmp/ShareGPT_V3_unfiltered_cleaned_split.json: 100%|█| 642M/642M [00:08<00:00, 8
[I 2026-11-15 10:14:55,908] A new study created in RDB with name: B-vllm-c8-throughput-tpe-median-s11


#Input tokens: 31744
#Output tokens: 7936
B-vllm-c8-throughput-tpe-median-s11: 1 existing trials, 20 remaining


2026-11-15 10:14:56,890 - llm_optimizer.server_utils - INFO - Starting server with command: vllm serve /tmp/hf-cache-08jual1n/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a --host 127.0.0.1 --port 8000 --max-num-seqs=72 --max-num-batched-tokens=16896 --gpu-memory-utilization=0.85 --enable-chunked-prefill '--compilation-config={"cudagraph_capture_sizes": [1, 2, 4, 8, 16, 32, 72], "cudagraph_mode": "FULL_AND_PIECEWISE"}' --dtype=bfloat16 --kv-cache-dtype=fp8 --performance-mode=interactivity --attention-backend=FLASHINFER --quantization=fp8 --calculate-kv-scales --no-async-scheduling --no-enable-prefix-caching
2026-11-15 10:14:56,892 - llm_optimizer.server_utils - INFO - Waiting for server to be ready at http://127.0.0.1:8000/health...
W1115 10:14:59.747000 24979 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1115 10:14:59.748000 24979 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use

[W 2026-11-15 10:15:11,947] Trial 0 failed with parameters: {'max_num_seqs': 72, 'max_num_batched_tokens': 16896, 'gpu_memory_utilization': 0.85, 'async_scheduling': False, 'performance_mode': 'interactivity', 'attention_backend': 'FLASHINFER', 'enable_prefix_caching': False, 'graph_mode': 'FULL_AND_PIECEWISE', 'graph_capture_profile': 'powers', 'weight_precision': 'fp8', 'kv_cache_precision': 'fp8'} because of the following error: BenchmarkFailure('ServerNotReadyError: Server did not become ready in the allotted time.').
Traceback (most recent call last):
  File "/home/ubuntu/llm-optimizer-trt/src/llm_optimizer/experiment/lab.py", line 606, in execute
    result, _ = run_single_configuration(
                ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/llm-optimizer-trt/src/llm_optimizer/cli.py", line 333, in run_single_configuration
    server_process = start_server(
                     ^^^^^^^^^^^^^
  File "/home/ubuntu/llm-optimizer-trt/src/llm_optimizer/server_utils.py", line 2

2026-11-15 10:15:12,093 - llm_optimizer.server_utils - INFO - Starting server with command: vllm serve /tmp/hf-cache-08jual1n/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a --host 127.0.0.1 --port 8000 --max-num-seqs=88 --max-num-batched-tokens=1536 --gpu-memory-utilization=0.75 --enable-chunked-prefill '--compilation-config={"cudagraph_capture_sizes": [1, 2, 4, 8, 16, 32, 88], "cudagraph_mode": "PIECEWISE"}' --dtype=bfloat16 --kv-cache-dtype=fp8 --performance-mode=interactivity --attention-backend=auto --quantization=fp8 --calculate-kv-scales --no-async-scheduling --no-enable-prefix-caching
2026-11-15 10:15:12,094 - llm_optimizer.server_utils - INFO - Waiting for server to be ready at http://127.0.0.1:8000/health...
W1115 10:15:14.930000 25046 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1115 10:15:14.930000 25046 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom

[W 2026-11-15 10:15:22,118] Trial 1 failed with parameters: {'max_num_seqs': 88, 'max_num_batched_tokens': 1536, 'gpu_memory_utilization': 0.75, 'async_scheduling': False, 'performance_mode': 'interactivity', 'attention_backend': 'auto', 'enable_prefix_caching': False, 'graph_mode': 'PIECEWISE', 'graph_capture_profile': 'powers', 'weight_precision': 'fp8', 'kv_cache_precision': 'fp8'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ubuntu/.venv-vllm/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/ubuntu/llm-optimizer-trt/src/llm_optimizer/experiment/lab.py", line 1441, in objective
    m = benchmark.measure(
        ^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/llm-optimizer-trt/src/llm_optimizer/experiment/lab.py", line 692, in measure
    return pool.submit(execute).result()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "

cost/cumulative_gpu_seconds,▁
parameters/gpu_memory_utilization,▁
parameters/max_num_batched_tokens,▁
parameters/max_num_seqs,▁
trial/gpu_seconds,▁
trial/number,▁
trial/orchestrator_wall_seconds,▁
cost/cumulative_gpu_seconds,15.01556
interrupted,True
parameters/async_scheduling,False
parameters/attention_backend,FLASHINFER


KeyboardInterrupt: 

## Independent candidate validation
Run after search, within the remaining GPU budget. This uses the repository’s shortlist and independent validation workload/repeats; it is not an automatic sustained-load soak test. Keep the serving stack pinned and record any separate soak workload. Validation in this interactive cell does not use the batch watchdog.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    from llm_optimizer.experiment.workflow import load_studies, write_json
    runner = get_runner()
    with runner.lock():
        studies = load_studies(runner.lab)
        validation = runner.lab.validate_candidates(runner.lab.shortlist(studies))
        write_json(runner.run_dir / "validation.json", runner.lab.artifact_value(validation))


## Unattended execution
Run `python -m llm_optimizer.experiment.phase1_experiments` from the repository root. It runs each group in a fresh process group, terminates that entire group on timeout, and retries at most `MAX_RESTARTS` times. It records a persistent campaign deadline and skips completed tasks on restart. Nonzero final exit means one or more groups exhausted retries. Review failures instead of treating them as valid measurements.

Example systemd service (replace user, checkout and environment paths):
```ini
[Unit]
Description=Phase 1 GPU experiments
After=network-online.target
StartLimitIntervalSec=3600
StartLimitBurst=5
[Service]
Type=simple
User=ubuntu
WorkingDirectory=/workspace/llm-optimizer
ExecStart=/workspace/llm-optimizer/.venv/bin/python -m llm_optimizer.experiment.phase1_experiments
Restart=on-failure
RestartSec=30
KillMode=control-group
TimeoutStopSec=30
Environment=PYTHONUNBUFFERED=1
Environment=CUDA_VISIBLE_DEVICES=0
[Install]
WantedBy=multi-user.target
```
Install as `/etc/systemd/system/phase1.service`, then run `sudo systemctl enable --now phase1`. Inspect `journalctl -u phase1 -f`. Keep clocks synchronized, durable disks attached, and artifacts backed up. The persisted 168-hour deadline survives service restarts. Transient failures and OOMs remain possible; the aim is preserved progress and bounded recovery.


## Optional Pareto, constraint, and pruning experiments
Run these cells independently after editing SLOs, samplers and pinned serving settings. Active pruning is supported only for unconstrained scalar objectives. Calibration uses the same task workload and scaled request checkpoints; pilot samples are additional to the 20-attempt comparison budgets. Diagnostics replay a first-checkpoint median rule and estimate savings; validate them on the GPU before trusting pruning. Threshold searches reuse a persisted task calibration.


In [ ]:
pareto_latency = run_objective_group("B", "pareto_latency", [1])
plot_group(pareto_latency)


In [ ]:
pareto_loaded = run_objective_group("B", "pareto_throughput", [c for c in CONCURRENCIES if c > 1])
plot_group(pareto_loaded)


In [ ]:
constrained_scalar = run_objective_group("B", "e2e", [1], constrained=True)
plot_group(constrained_scalar)


In [ ]:
constrained_pareto = run_objective_group("B", "pareto_latency", [1], constrained=True)
plot_group(constrained_pareto)


In [ ]:
diagnostics = get_runner().calibrate("B", "throughput", 8, FRAMEWORKS[0], samples=PRUNING_SAMPLES)
diagnostics


In [ ]:
pruning_comparison = {name: run_objective_group("B", "throughput", [8], pruner_name=name)
                      for name in PRUNER_NAMES}
